In [ ]:
# ===== OLMo3 独立实现 Notebook：环境检查 =====
# 本 notebook 不依赖 transformers 库，从零手写实现 OLMo3（AllenAI 开源模型系列）的
# 模型结构、权重加载与文本生成流程。这里先检查关键依赖包（权重下载 / 分词 / 张量计算）的版本。
from importlib.metadata import version

pkgs = [
    "huggingface_hub",  # to download pretrained weights
    "tokenizers",       # to implement the tokenizer
    "torch",            # to implement the model
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
# ===== 选择要加载的 OLMo3 模型变体 =====
# OLMo3 命名规则示例：Olmo-3-1025-7B 中的 1025/1125 是发布日期后缀（2025年10月/11月）的基础/预训练权重；
# 带 Instruct 后缀是指令微调版本，带 Think 后缀是长思维链(CoT)推理版本，
# 带 RLZero-IF 后缀是经过 RLZero 强化学习(且强调指令遵循 IF)训练的版本；
# 7B / 32B 对应不同参数规模，分别使用下面的 OLMO3_CONFIG_7B / OLMO3_CONFIG_32B 配置。
# 取消注释其中一行、注释掉其余行即可切换要下载和运行的模型。
# Select which model to use

# USE_MODEL = "Olmo-3-1025-7B"
# USE_MODEL = "Olmo-3-1125-32B"
USE_MODEL = "Olmo-3-7B-Instruct"
# USE_MODEL = "Olmo-3-32B-Instruct"
# USE_MODEL = "Olmo-3-7B-Think"
# USE_MODEL = "Olmo-3-32B-Think"
# USE_MODEL = "Olmo-3-7B-RLZero-IF"

1. Architecture code

In [ ]:
# ============================================================
# 第一部分：OLMo3 模型结构定义
# 依次实现：SwiGLU 前馈网络 -> RMSNorm -> RoPE（含 YaRN 长上下文扩展）->
# 分组查询注意力 GQA（并带 QK-Norm）-> Transformer Block（后置归一化 Post-Norm 结构）->
# 整体 Olmo3Model（局部滑窗注意力与全局注意力交替的混合注意力模式）。
# ============================================================
import torch
import torch.nn as nn


# ---- SwiGLU 前馈网络（FFN）----
# OLMo3 沿用 Llama / OLMo2 等模型的 SwiGLU 结构：用两路独立的线性层
# （fc1 对应 HF 权重里的 gate_proj，fc2 对应 up_proj）把输入从 emb_dim 升维到 hidden_dim，
# 其中一路经过 SiLU 激活后与另一路逐元素相乘做门控，再经 fc3（对应 down_proj）投影回 emb_dim。
# 三个线性层都不带 bias，这是现代大模型的常见做法。
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], dtype=cfg["dtype"], bias=False)

    # 前向传播：x 形状为 (batch, seq_len, emb_dim)
    def forward(self, x):
        x_fc1 = self.fc1(x)
        x_fc2 = self.fc2(x)
        # SwiGLU 门控：SiLU(fc1(x)) * fc2(x)，两路输出的形状均为 (batch, seq_len, hidden_dim)
        x = nn.functional.silu(x_fc1) * x_fc2
        return self.fc3(x)

# ---- RMSNorm（Root Mean Square Layer Normalization）----
# 与 LayerNorm 不同，RMSNorm 不做均值居中(mean-centering)，只用均方根做缩放归一化，
# 计算量更小，在大模型中效果与 LayerNorm 相当，是 Llama/OLMo 系列的标准归一化方式。
class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(emb_dim))

    # x 形状: (..., emb_dim)；先转换成 float32 计算方差以提升数值稳定性，最后再转换回原始 dtype（如 bfloat16）
    def forward(self, x):
        input_dtype = x.dtype
        x_f = x.float()
        var = x_f.pow(2).mean(dim=-1, keepdim=True)
        # RMS 归一化核心公式：x / sqrt(mean(x^2) + eps)，用 rsqrt 代替除法以提高效率
        x_norm = x_f * torch.rsqrt(var + self.eps)
        return (self.weight * x_norm).to(input_dtype)

# ---- RoPE（Rotary Position Embedding，旋转位置编码）----
# RoPE 通过对 Q/K 向量做逐维旋转来注入位置信息，不需要额外的位置向量与输入相加。
# OLMo3 支持两种模式：
#   1) 默认 RoPE：直接按训练时的原始上下文长度计算旋转频率；
#   2) YaRN（Yet another RoPE extensioN）：对不同频率维度做插值/外推混合缩放，
#      把预训练时较短的上下文长度（rope_orig_max）扩展到推理时更长的上下文（如 65536）。
import math


def compute_rope_params(head_dim, theta_base=10_000, context_length=4096, attention_factor=1.0, rope_type="default", rope_factor=1.0, rope_orig_max=8192, beta_fast=32.0, beta_slow=1.0, dtype=torch.float32):
    assert head_dim % 2 == 0, "Embedding dimension must be even"

    # YaRN 模式：用于长上下文扩展（把训练时 rope_orig_max=8192 的位置编码外推到更大的 context_length）
    if rope_type == "yarn":
        # Compute YaRN-style frequency scaling (as per https://huggingface.co/papers/2309.00071)

        # 计算「旋转次数 num_rotations」对应的频率维度下标（YaRN 论文中的修正维度公式）
        def find_correction_dim(num_rotations, dim, base, max_position_embeddings):
            """Inverse dimension formula to find the dimension based on the number of rotations"""
            return (dim * math.log(max_position_embeddings / (num_rotations * 2 * math.pi))) / (2 * math.log(base))

        # 根据 beta_fast / beta_slow 两个超参数，找到需要在插值和外推之间做混合过渡的维度区间
        def find_correction_range(low_rot, high_rot, dim, base, max_position_embeddings):
            """Find dimension range bounds based on rotations"""
            low = find_correction_dim(low_rot, dim, base, max_position_embeddings)
            high = find_correction_dim(high_rot, dim, base, max_position_embeddings)
            low = math.floor(low)
            high = math.ceil(high)
            return max(low, 0), min(high, dim - 1)

        # 在 [min_val, max_val] 区间内生成一个从 0 平滑过渡到 1 的斜坡权重，用于混合插值/外推频率
        def linear_ramp_factor(min_val, max_val, dim):
            if min_val == max_val:
                max_val += 0.001  # Prevent singularity
            linear_func = (torch.arange(dim, dtype=torch.float32) - min_val) / (max_val - min_val)
            ramp_func = torch.clamp(linear_func, 0, 1)
            return ramp_func

        # 计算基础频率 inv_freq：不做长度缩放的“外推”版本 与 按 rope_factor 缩放的“插值”版本
        # Base frequencies
        pos_freqs = theta_base ** (torch.arange(0, head_dim, 2, dtype=dtype) / head_dim)
        inv_freq_extrapolation = 1.0 / pos_freqs  # No scaling (extrapolation)
        inv_freq_interpolation = 1.0 / (rope_factor * pos_freqs)  # With scaling (interpolation)

        # 找到低频/高频修正区间，用于决定各维度上外推和插值的混合比例
        # Find the range where we blend between interpolation and extrapolation
        low, high = find_correction_range(beta_fast, beta_slow, head_dim, theta_base, rope_orig_max)

        # 计算每个维度的“外推因子”：越靠近高频维度越倾向纯外推，越靠近低频维度越倾向纯插值
        # Get n-dimensional rotational scaling corrected for extrapolation
        inv_freq_extrapolation_factor = 1 - linear_ramp_factor(low, high, head_dim // 2).to(dtype=dtype)
        inv_freq = (
            inv_freq_interpolation * (1 - inv_freq_extrapolation_factor)
            + inv_freq_extrapolation * inv_freq_extrapolation_factor
        )
    # 非 YaRN 情况：标准 RoPE，频率完全按 theta_base 的幂次衰减，不做长度缩放
    else:
        # Default RoPE
        inv_freq = 1.0 / (
            theta_base ** (
                torch.arange(0, head_dim, 2, dtype=dtype)[: head_dim // 2].float()
                / head_dim
            )
        )

    # 为每个绝对位置 0..context_length-1 生成位置索引
    # Generate position indices
    positions = torch.arange(context_length, dtype=dtype)

    # 角度 = 位置 × 频率，得到每个位置在每个频率维度上的旋转角，形状 (context_length, head_dim//2)
    # Compute the base angles (shape: [context_length, head_dim // 2])
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)

    # 把 head_dim//2 个角度复制拼接成完整的 head_dim，方便后续对整个 head_dim 做旋转
    # Expand to full head_dim (shape: [context_length, head_dim])
    angles = torch.cat([angles, angles], dim=1)

    # 预先计算 cos/sin 表；attention_factor 是 YaRN 中用来补偿注意力“温度”的缩放系数
    # Precompute sine and cosine
    cos = torch.cos(angles) * attention_factor
    sin = torch.sin(angles) * attention_factor

    return cos, sin



# apply_rope: 对 Q/K 张量应用旋转位置编码
# x 形状: (batch_size, num_heads, seq_len, head_dim)
def apply_rope(x, cos, sin):
    # x: (batch_size, num_heads, seq_len, head_dim)
    batch_size, num_heads, seq_len, head_dim = x.shape
    assert head_dim % 2 == 0, "Head dimension must be even"

    # 把 head_dim 切成前后两半，用于构造“旋转后”的向量（rotate_half 技巧）
    # Split x into first half and second half
    x1 = x[..., : head_dim // 2]  # First half
    x2 = x[..., head_dim // 2 :]  # Second half

    # 按当前序列长度截取 cos/sin，并广播成 (1, 1, seq_len, head_dim) 以便与 x 逐元素相乘
    # Adjust sin and cos shapes
    cos = cos[:seq_len, :].unsqueeze(0).unsqueeze(0)  # Shape: (1, 1, seq_len, head_dim)
    sin = sin[:seq_len, :].unsqueeze(0).unsqueeze(0)

    # RoPE 旋转公式：x_rotated = x*cos + rotate_half(x)*sin，其中 rotate_half(x) = concat(-x2, x1)
    # Apply the rotary transformation
    rotated = torch.cat((-x2, x1), dim=-1)
    x_rotated = (x * cos) + (rotated * sin)

    # It's ok to use lower-precision after applying cos and sin rotation
    return x_rotated.to(dtype=x.dtype)

# ---- 分组查询注意力（Grouped-Query Attention, GQA）----
# GQA 是多头注意力（MHA）与多查询注意力（MQA）之间的折中：Q 仍使用 num_heads 个头，
# 但 K/V 只用较少的 num_kv_groups 组，多个 Q 头共享同一组 K/V，从而显著减少推理时的 KV 缓存显存占用。
# OLMo3 的特色（QK-Norm）：在 Q、K 投影之后、reshape 成多头之前，先对整个投影结果做一次 RMSNorm，
# 用于稳定注意力的数值范围；这是对“拼接后的全部维度”做 norm，不同于按单个 head 分别做 norm 的方案。
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_in, num_heads, num_kv_groups, head_dim, attention_bias=False, dtype=None, sliding_window=None, attn_type="full_attention"):
        super().__init__()
        assert num_heads % num_kv_groups == 0, "num_heads must be divisible by num_kv_groups"

        self.num_heads = num_heads
        self.num_kv_groups = num_kv_groups
        self.group_size = num_heads // num_kv_groups

        self.head_dim = head_dim
        self.d_out = num_heads * head_dim
        self.attn_type = attn_type
        # 只有滑窗注意力(sliding_attention)层才会真正用到 sliding_window 窗口大小；
        # 全局注意力(full_attention)层的 sliding_window 设为 None，表示不做局部窗口限制
        self.sliding_window = sliding_window if attn_type == "sliding_attention" else None

        # Projections
        self.W_query = nn.Linear(d_in, self.d_out, bias=attention_bias, dtype=dtype)
        self.W_key = nn.Linear(d_in, num_kv_groups * head_dim, bias=attention_bias, dtype=dtype)
        self.W_value = nn.Linear(d_in, num_kv_groups * head_dim, bias=attention_bias, dtype=dtype)
        self.out_proj = nn.Linear(self.d_out, d_in, bias=attention_bias, dtype=dtype)

        # q_norm 作用于展平后的 Q 投影（维度 = num_heads*head_dim），
        # k_norm 作用于展平后的 K 投影（维度 = num_kv_groups*head_dim）
        # Olmo3-style RMSNorm over the flattened projections
        self.q_norm = RMSNorm(self.d_out)
        self.k_norm = RMSNorm(num_kv_groups * head_dim)

    # x: (batch, seq_len, d_in)；mask 由外层按当前层类型（全局/局部）传入；cos/sin 为预计算的 RoPE 表
    def forward(self, x, mask, cos, sin):
        b, num_tokens, _ = x.shape

        # Apply projections
        # 线性投影：Q 用 num_heads 个头的维度，K/V 只用 num_kv_groups 个头的维度（远小于 num_heads）
        queries = self.W_query(x)  # (b, num_tokens, num_heads * head_dim)
        keys = self.W_key(x)       # (b, num_tokens, num_kv_groups * head_dim)
        values = self.W_value(x)   # (b, num_tokens, num_kv_groups * head_dim)

        # 对完整的 Q/K 投影向量做 QK-Norm（OLMo3 特有），提升注意力数值稳定性
        # Normalize q and k
        queries = self.q_norm(queries)
        keys = self.k_norm(keys)

        # 拆分成多头：(b, num_tokens, num_heads*head_dim) -> (b, num_heads, num_tokens, head_dim)
        # Reshape to (b, heads, seq, head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys = keys.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)
        values = values.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)

        # 对 Q、K 分别注入旋转位置编码（V 不需要位置编码）
        # Apply RoPE
        queries = apply_rope(queries, cos, sin)
        keys = apply_rope(keys, cos, sin)

        # 把每组 K/V 沿 head 维度重复 group_size 次，扩展成与 Q 相同的 num_heads，
        # 这样才能让多个 Q 头共享同一组 K/V 逐头做注意力矩阵乘法
        # Expand KV groups to full head count
        if self.group_size > 1:
            keys = keys.repeat_interleave(self.group_size, dim=1)
            values = values.repeat_interleave(self.group_size, dim=1)

        # 注意：这里先把 queries 乘以缩放系数 1/sqrt(head_dim)，再做 QK^T，
        # 而不是像常规实现那样在算出 attn_scores 之后再统一缩放；对 Olmo 系列数值更稳定
        # Scaling before the matmul seems to be a bit more stable for Olmo
        scale = self.head_dim ** -0.5  # Python float
        queries = queries * scale

        # 计算注意力得分，形状 (b, num_heads, num_tokens, num_tokens)；
        # 用 mask 屏蔽掉不该被看到的位置（设为 -inf，softmax 后趋近于 0）
        # Attention
        attn_scores = queries @ keys.transpose(2, 3)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask, -torch.inf)

        # softmax 得到注意力权重，与 V 加权求和后拼回 (b, num_tokens, d_out)，再做输出投影
        attn_weights = torch.softmax(attn_scores, dim=-1)
        context = (attn_weights @ values).transpose(1, 2).reshape(b, num_tokens, self.d_out)
        return self.out_proj(context)

# ---- Transformer Block：OLMo3 使用「后置归一化」(Post-Norm) 结构 ----
# 与 GPT/Llama 常见的 Pre-Norm（先 norm 再进子层）不同，这里是：先算注意力/前馈的输出，
# 再对该输出做 RMSNorm，最后才和残差(shortcut)相加，即 x = x + Norm(SubLayer(x))。
# 每个 block 的注意力类型(attn_type)由外部的 layer_types 配置决定，用来在
# 滑窗局部注意力(sliding_attention)与全局注意力(full_attention)之间切换。
class TransformerBlock(nn.Module):
    def __init__(self, cfg, attn_type):
        super().__init__()
        self.attn_type = attn_type
        self.att = GroupedQueryAttention(
            d_in=cfg["emb_dim"],
            num_heads=cfg["n_heads"],
            num_kv_groups=cfg["n_kv_heads"],
            head_dim=cfg["head_dim"],
            attention_bias=cfg["attention_bias"],
            dtype=cfg["dtype"],
            sliding_window=cfg["sliding_window"],
            attn_type=attn_type,
        )
        self.ff = FeedForward(cfg)
        self.post_attention_layernorm = RMSNorm(cfg["emb_dim"], eps=cfg["rms_norm_eps"])
        self.post_feedforward_layernorm = RMSNorm(cfg["emb_dim"], eps=cfg["rms_norm_eps"])

    # 根据本层的 attn_type 选择使用局部滑窗掩码还是全局因果掩码
    def forward(self, x, mask_global, mask_local, cos, sin):
        attn_mask = mask_local if self.attn_type == "sliding_attention" else mask_global

        # 注意力子层：post-norm 结构 —— 先算注意力，再做 RMSNorm，再加残差
        shortcut = x
        x_attn = self.att(x, attn_mask, cos, sin)
        x_attn = self.post_attention_layernorm(x_attn)
        x = shortcut + x_attn

        # 前馈子层：同样是 post-norm 结构 —— 先算 SwiGLU FFN，再做 RMSNorm，再加残差
        shortcut = x
        x_ffn = self.ff(x)
        x_ffn = self.post_feedforward_layernorm(x_ffn)
        x = shortcut + x_ffn
        return x

# ---- OLMo3 整体模型 ----
# 关键特性：
#  1) 混合注意力模式：多数层用滑动窗口注意力(sliding_attention，只关注局部上下文、降低 KV 缓存开销)，
#     每 4 层中有 1 层用全局注意力(full_attention，可关注任意历史位置，弥补长距离依赖)；
#  2) 支持 YaRN 长上下文扩展的 RoPE；
#  3) 整体使用 RMSNorm + SwiGLU + GQA(+QK-Norm) + Post-Norm 残差结构。
class Olmo3Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg["layer_types"] is not None and len(cfg["layer_types"]) == cfg["n_layers"]

        # 词嵌入表：(vocab_size, emb_dim)，把 token id 映射为向量
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])
        # 按 cfg["layer_types"] 逐层构造 TransformerBlock，每层的注意力类型可能不同（局部/全局交替）
        self.blocks = nn.ModuleList([TransformerBlock(cfg, attn_type) for attn_type in cfg["layer_types"]])
        self.final_norm = RMSNorm(cfg["emb_dim"], eps=cfg["rms_norm_eps"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])
        self.cfg = cfg

        # 预先计算好整段 context_length 长度的 RoPE cos/sin 表
        cos, sin = compute_rope_params(
            head_dim=cfg["head_dim"],
            context_length=cfg["context_length"],
            theta_base=cfg["rope_base"],
            attention_factor=cfg["rope_attention_factor"],
            rope_type=cfg["rope_type"],
            rope_factor=cfg["rope_factor"],
            rope_orig_max=cfg["rope_orig_max"],
            dtype=torch.float32,
        )
        # 注册为 buffer（不参与梯度更新、persistent=False 表示不写入 state_dict），
        # ⚠️ 风险点标注：上面调用 compute_rope_params 时没有把 cfg["beta_fast"]/cfg["beta_slow"]
        # 传进去，而是依赖该函数的默认参数 beta_fast=32.0, beta_slow=1.0；目前恰好与
        # OLMO3_CONFIG 中的取值一致，但如果以后配置里的 beta_fast/beta_slow 被改动，
        # 这里不会同步生效，存在潜在的不一致风险，建议关注（未做修改，仅标注）。
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)

    # 构造两种注意力掩码：全局因果掩码(mask_global) 和 滑窗局部因果掩码(mask_local)
    # 掩码中 True 表示该位置需要被屏蔽（不能被注意力看到）
    def create_masks(self, seq_len, device):
        ones = torch.ones((seq_len, seq_len), dtype=torch.bool, device=device)

        # mask_global (future is masked: j > i)
        #     j:  0 1 2 3 4 5 6 7
        #  i
        #     0:  0 1 1 1 1 1 1 1
        #     1:  0 0 1 1 1 1 1 1
        #     2:  0 0 0 1 1 1 1 1
        #     3:  0 0 0 0 1 1 1 1
        #     4:  0 0 0 0 0 1 1 1
        #     5:  0 0 0 0 0 0 1 1
        #     6:  0 0 0 0 0 0 0 1
        #     7:  0 0 0 0 0 0 0 0
        # 标准因果掩码：上三角(不含对角线)为 True，即每个位置只能看到自己和之前的位置
        mask_global = torch.triu(ones, diagonal=1)

        # far_past (too far back is masked: i - j >= sliding_window)
        # where sliding_window = 4
        #     j:  0 1 2 3 4 5 6 7
        #  i
        #     0:  0 0 0 0 0 0 0 0
        #     1:  0 0 0 0 0 0 0 0
        #     2:  0 0 0 0 0 0 0 0
        #     3:  0 0 0 0 0 0 0 0
        #     4:  1 0 0 0 0 0 0 0
        #     5:  1 1 0 0 0 0 0 0
        #     6:  1 1 1 0 0 0 0 0
        #     7:  1 1 1 1 0 0 0 0
        # 滑窗之外“太久远的过去”也要屏蔽：结合 sliding_window 大小，构造下三角形式的远古位置掩码
        far_past = torch.triu(ones, diagonal=self.cfg["sliding_window"]).T

        # Local (sliding_window) = future OR far-past
        # mask_local
        #     j:  0 1 2 3 4 5 6 7
        # i
        # 0:      0 1 1 1 1 1 1 1
        # 1:      0 0 1 1 1 1 1 1
        # 2:      0 0 0 1 1 1 1 1
        # 3:      0 0 0 0 1 1 1 1
        # 4:      1 0 0 0 0 1 1 1
        # 5:      1 1 0 0 0 0 1 1
        # 6:      1 1 1 0 0 0 0 1
        # 7:      1 1 1 1 0 0 0 0
        # 局部注意力掩码 = 未来位置 OR 太久远的过去位置，两者取并集（只保留窗口内的历史 token 可见）
        mask_local = mask_global | far_past
        return mask_global, mask_local

    # 整体前向：input_ids 形状 (batch, seq_len) -> logits 形状 (batch, seq_len, vocab_size)
    def forward(self, input_ids):
        b, seq_len = input_ids.shape
        # 词嵌入: (b, seq_len) -> (b, seq_len, emb_dim)
        x = self.tok_emb(input_ids)
        # 为当前序列长度动态构造两种掩码
        mask_global, mask_local = self.create_masks(seq_len, x.device)

        # 截取当前序列长度所需的 RoPE cos/sin
        cos = self.cos[:seq_len, :].to(x.device)
        sin = self.sin[:seq_len, :].to(x.device)

        # 依次通过每一层 TransformerBlock，各层按自己的 attn_type 使用对应的掩码
        for block in self.blocks:
            x = block(x, mask_global, mask_local, cos, sin)

        # 最终归一化
        x = self.final_norm(x)
        # 输出投影到词表维度: (b, seq_len, emb_dim) -> (b, seq_len, vocab_size)；
        # 显式转换回 cfg["dtype"]（如 bfloat16）以保证与权重 dtype 一致
        logits = self.out_head(x.to(self.cfg["dtype"]))
        return logits

2. Initialize model

In [ ]:
# ============================================================
# 第二部分：初始化模型 —— OLMo3 7B 模型配置（对应 allenai/Olmo-3-7B-* 系列 checkpoint）
# ============================================================
OLMO3_CONFIG_7B = {
    # 词表大小（对应 OLMo3 使用的 BPE 分词器）
    "vocab_size": 100_278,
    # 最大上下文长度：通过 YaRN 从预训练时的 rope_orig_max=8192 扩展到 65536
    "context_length": 65_536,
    "emb_dim": 4_096,
    # 注意力头数（Query 头数）
    "n_heads": 32,
    "n_layers": 32,
    "hidden_dim": 11_008,
    "head_dim": 128,
    # KV 头组数：7B 版本 n_kv_heads == n_heads，即退化为普通多头注意力(MHA)，未使用 GQA 压缩
    "n_kv_heads": 32,
    "attention_bias": False,
    "attention_dropout": 0.0,
    # 滑动窗口大小：sliding_attention 层只能看到最近 4096 个 token
    "sliding_window": 4_096,
    # 每层的注意力类型：按“3 层 sliding_attention + 1 层 full_attention”循环，
    # 混合局部/全局注意力，兼顾长距离依赖建模与推理时的显存/计算开销
    "layer_types": [
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
    ],
    # RoPE 基础频率 theta_base（值越大，位置编码对长距离位置的区分度越细）
    "rope_base": 500_000.0,
    "rope_attention_factor": 1.2079441541679836,
    # 使用 YaRN 方式做 RoPE 长上下文外推（而不是默认 RoPE）
    "rope_type": "yarn",
    "rope_factor": 8.0,
    "rope_orig_max": 8_192,
    # YaRN 的插值/外推混合区间超参数（对应 compute_rope_params 中的 find_correction_range）
    "beta_fast": 32.0,
    "beta_slow": 1.0,
    "rms_norm_eps": 1e-6,
    # 模型权重与计算精度：bfloat16（大模型推理常用精度，兼顾显存与数值范围）
    "dtype": torch.bfloat16,
    # 结束符/填充符 token id；若分词器文件能解析出对应特殊符号，OlmoTokenizer 会优先使用解析结果
    "eos_token_id": 100_257,
    "pad_token_id": 100_277,
}


# ============================================================
# OLMo3 32B 模型配置（对应 allenai/Olmo-3-32B-* 系列 checkpoint）
# 与 7B 版本的主要差异：更大的 emb_dim/n_layers/hidden_dim，
# 且真正使用了 GQA（n_kv_heads=8 远小于 n_heads=40）
# ============================================================
OLMO3_CONFIG_32B = {
    "vocab_size": 100_278,
    "context_length": 65_536,
    "emb_dim": 5_120,
    "n_heads": 40,
    "n_layers": 64,
    "hidden_dim": 27_648,
    "head_dim": 128,
    # KV 头组数远小于 n_heads(40)：32B 版本真正使用 GQA，
    # 每组 KV 被 40/8=5 个 Q 头共享，显著节省 KV 缓存显存
    "n_kv_heads": 8,
    "attention_bias": False,
    "attention_dropout": 0.0,
    "sliding_window": 4_096,
    "layer_types": [
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
    ],
    "rope_base": 500_000.0,
    "rope_attention_factor": 1.2079441541679836,
    "rope_type": "yarn",
    "rope_factor": 8.0,
    "rope_orig_max": 8_192,
    "beta_fast": 32.0,
    "beta_slow": 1.0,
    "rms_norm_eps": 1e-6,
    "dtype": torch.bfloat16,
    "eos_token_id": 100_257,
    "pad_token_id": 100_277,
}


# 根据前面选择的 USE_MODEL 名称中是否包含 "32B" 来选用对应的配置
OLMO3_CONFIG = OLMO3_CONFIG_32B if "32B" in USE_MODEL else OLMO3_CONFIG_7B
# 固定随机种子，保证权重加载前的随机初始化可复现（后续会被“3. Load pretrained weights”中的真实权重覆盖）
torch.manual_seed(123)
# 实例化模型（此时权重还是随机初始化，仅用于结构检查）
model = Olmo3Model(OLMO3_CONFIG)
model

In [ ]:
# 用一个假的输入 (batch=1, seq_len=3) 测试模型前向传播是否能跑通，输出 logits 形状应为 (1, 3, vocab_size)
model(torch.tensor([1, 2, 3]).unsqueeze(0))

In [ ]:
# 自动选择运行设备：优先 CUDA GPU，其次 Apple Silicon 的 MPS，最后回退到 CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model.to(device);

3. Load pretrained weights

In [ ]:
# ============================================================
# 第三部分：加载 HuggingFace 格式的预训练权重
# 把 safetensors 中以 HF 命名规则（如 model.layers.{l}.self_attn.q_proj.weight）存储的权重，
# 逐个拷贝进我们自己实现的 Olmo3Model 对应子模块的参数中。
# ============================================================
def load_weights_into_olmo(model, param_config, params):
    # 内部工具函数：把 right（HF 权重张量）安全地拷贝进 left（本模型的参数），
    # 会先检查形状是否一致，避免因命名映射写错而悄悄加载出错误形状的权重
    def assign(left, right, tensor_name="unknown"):
        if left.shape != right.shape:
            raise ValueError(
                f"Shape mismatch in tensor '{tensor_name}'. "
                f"Left: {left.shape}, Right: {right.shape}"
            )

        with torch.no_grad():
            if isinstance(right, torch.Tensor):
                left.copy_(right)
            else:
                left.copy_(torch.as_tensor(right, dtype=left.dtype, device=left.device))

        return left

    # 词嵌入权重：HF 命名为 model.embed_tokens.weight
    # Token embedding
    if "model.embed_tokens.weight" in params:
        model.tok_emb.weight = assign(
            model.tok_emb.weight,
            params["model.embed_tokens.weight"],
            "model.embed_tokens.weight",
        )

    # 逐层拷贝 Transformer Block 的权重
    for l in range(param_config["n_layers"]):
        block = model.blocks[l]
        att = block.att

        # 注意力的 Q/K/V 三个线性投影权重
        # Q, K, V projections
        att.W_query.weight = assign(
            att.W_query.weight,
            params[f"model.layers.{l}.self_attn.q_proj.weight"],
            f"model.layers.{l}.self_attn.q_proj.weight",
        )
        att.W_key.weight = assign(
            att.W_key.weight,
            params[f"model.layers.{l}.self_attn.k_proj.weight"],
            f"model.layers.{l}.self_attn.k_proj.weight",
        )
        att.W_value.weight = assign(
            att.W_value.weight,
            params[f"model.layers.{l}.self_attn.v_proj.weight"],
            f"model.layers.{l}.self_attn.v_proj.weight",
        )

        # 注意力输出投影（多头拼接后再投影回 emb_dim）
        # Output projection
        att.out_proj.weight = assign(
            att.out_proj.weight,
            params[f"model.layers.{l}.self_attn.o_proj.weight"],
            f"model.layers.{l}.self_attn.o_proj.weight",
        )

        # OLMo3 特有的 QK-Norm 权重（分别对应 Q、K 投影展平后的 RMSNorm 缩放参数）
        # QK norms
        att.q_norm.weight = assign(
            att.q_norm.weight,
            params[f"model.layers.{l}.self_attn.q_norm.weight"],
            f"model.layers.{l}.self_attn.q_norm.weight",
        )
        att.k_norm.weight = assign(
            att.k_norm.weight,
            params[f"model.layers.{l}.self_attn.k_norm.weight"],
            f"model.layers.{l}.self_attn.k_norm.weight",
        )

        # SwiGLU 前馈网络的三个权重：gate_proj -> fc1，up_proj -> fc2，down_proj -> fc3
        # Feedforward weights
        block.ff.fc1.weight = assign(
            block.ff.fc1.weight,
            params[f"model.layers.{l}.mlp.gate_proj.weight"],
            f"model.layers.{l}.mlp.gate_proj.weight",
        )
        block.ff.fc2.weight = assign(
            block.ff.fc2.weight,
            params[f"model.layers.{l}.mlp.up_proj.weight"],
            f"model.layers.{l}.mlp.up_proj.weight",
        )
        block.ff.fc3.weight = assign(
            block.ff.fc3.weight,
            params[f"model.layers.{l}.mlp.down_proj.weight"],
            f"model.layers.{l}.mlp.down_proj.weight",
        )

        # post-norm 结构下，注意力子层和前馈子层各自输出之后的 RMSNorm 权重
        # Post-attention and post norms
        block.post_attention_layernorm.weight = assign(
            block.post_attention_layernorm.weight,
            params[f"model.layers.{l}.post_attention_layernorm.weight"],
            f"model.layers.{l}.post_attention_layernorm.weight",
        )
        block.post_feedforward_layernorm.weight = assign(
            block.post_feedforward_layernorm.weight,
            params[f"model.layers.{l}.post_feedforward_layernorm.weight"],
            f"model.layers.{l}.post_feedforward_layernorm.weight",
        )

    # 最终归一化层权重 (model.norm.weight) 与语言模型输出头 (lm_head.weight)
    # Final normalization and output head
    if "model.norm.weight" in params:
        model.final_norm.weight = assign(
            model.final_norm.weight,
            params["model.norm.weight"],
            "model.norm.weight",
        )

    # 有些 checkpoint 会做「权重绑定」(weight tying)：输出头与词嵌入共享同一份权重矩阵，
    # 此时权重文件里不会单独存 lm_head.weight，需要退化为直接复用 tok_emb.weight
    if "lm_head.weight" in params:
        model.out_head.weight = assign(
            model.out_head.weight,
            params["lm_head.weight"],
            "lm_head.weight",
        )
    else:
        model.out_head.weight = model.tok_emb.weight
        print("Model uses weight tying.")

# ---- 从 HuggingFace Hub 下载 OLMo3 权重文件并加载 ----
import json
import os
from pathlib import Path
from safetensors.torch import load_file
from huggingface_hub import snapshot_download

# 根据前面选择的 USE_MODEL 拼出 HuggingFace Hub 上的仓库 id，例如 allenai/Olmo-3-7B-Instruct
repo_id = f"allenai/{USE_MODEL}"
local_dir = Path(repo_id).parts[-1]

# 下载整个仓库快照（模型权重分片、tokenizer 等文件）到本地目录
repo_dir = snapshot_download(repo_id=repo_id, local_dir=local_dir)
# 大模型权重通常被切分成多个 .safetensors 分片文件，index 文件记录了每个参数名存放在哪个分片里
index_path = os.path.join(repo_dir, "model.safetensors.index.json")
with open(index_path, "r") as f:
    index = json.load(f)

# 依次加载每个分片文件，合并成一个「参数名 -> 张量」的完整字典
weights_dict = {}
for filename in sorted(set(index["weight_map"].values())):
    shard_path = os.path.join(repo_dir, filename)
    shard = load_file(shard_path)
    weights_dict.update(shard)

# 把下载好的权重字典按照 HF 命名规则映射进我们自己实现的模型结构中
load_weights_into_olmo(model, OLMO3_CONFIG, weights_dict)
# 加载完权重后再显式挪到目标设备（GPU/MPS/CPU）
model.to(device)
del weights_dict

4. Load tokenizer

In [ ]:
# ============================================================
# 第四部分：加载分词器并构造对话模板
# 使用 HuggingFace 的 tokenizers 库直接加载 tokenizer.json（BPE 分词器），
# 并用 ChatML 风格的 <|im_start|>/<|im_end|> 标记拼出模型期望的对话输入格式。
# ============================================================
from tokenizers import Tokenizer
from huggingface_hub import hf_hub_download


# 对 tokenizers.Tokenizer 的简单封装，统一提供 encode/decode 接口，并解析 eos/pad 特殊符号
class OlmoTokenizer:
    def __init__(self, tokenizer_file_path, eos_token_id, pad_token_id):
        tok_file = Path(tokenizer_file_path)
        self._tok = Tokenizer.from_file(str(tok_file))
        # 优先从分词器词表里查找真实存在的结束符（不同 checkpoint 用的结束符名字可能不同）；
        # ⚠️ 风险点标注：这里用 `or` 短路查找，如果某个候选 token 的 id 恰好是 0，
        # 会被 Python 当作 False 而被跳过、继续查下一个候选，属于潜在边界 bug，
        # 但对当前 OLMo3 分词器（结束符 id 通常不为 0）不会触发，故只标注不修改
        eos_from_tok = (
            self._tok.token_to_id("<|endoftext|>")
            or self._tok.token_to_id("<end_of_turn>")
        )
        # 找不到就退回使用配置文件里给的 eos_token_id
        self.eos_token_id = eos_from_tok if eos_from_tok is not None else eos_token_id
        # 同理查找填充符（存在与上面相同的 “id 恰好为 0 会被误跳过” 的潜在风险）
        pad_from_tok = (
            self._tok.token_to_id("<|pad|>")
            or self._tok.token_to_id("<pad>")
        )
        self.pad_token_id = pad_from_tok if pad_from_tok is not None else pad_token_id

    # 文本 -> token id 列表
    def encode(self, text):
        return self._tok.encode(text).ids

    # token id 列表 -> 文本；skip_special_tokens=False 保留特殊符号，便于观察模型是否正确生成了结束符
    def decode(self, ids):
        return self._tok.decode(ids, skip_special_tokens=False)



# 构造 OLMo3（Instruct/Think 系列）使用的 ChatML 风格对话模板：
# <|im_start|>user\n...\n<|im_end|>\n<|im_start|>assistant\n
# 模型会在 assistant 段之后续写回复
def apply_chat_template(user_text):
    return (
        "<|im_start|>user\n"
        f"{user_text}\n"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
    )


# 优先使用前面 snapshot_download 已下载好的 tokenizer.json，若不存在则单独下载
tokenizer_file_path = os.path.join(local_dir, "tokenizer.json")
if not os.path.exists(tokenizer_file_path):
    try:
        tokenizer_file_path = hf_hub_download(repo_id=repo_id, filename="tokenizer.json", local_dir=local_dir)
    except Exception as e:
        print(f"Warning: failed to download tokenizer.json: {e}")
        tokenizer_file_path = "tokenizer.json"

# 构造分词器实例
tokenizer = OlmoTokenizer(
    tokenizer_file_path=tokenizer_file_path,
    eos_token_id=OLMO3_CONFIG["eos_token_id"],
    pad_token_id=OLMO3_CONFIG["pad_token_id"],
)
# 用对话模板包装用户提问，得到最终喂给模型的 prompt 字符串
prompt = apply_chat_template("Give me a short intro to large language models in 3 sentences.")

# 编码成 token id 列表（一维 Python list），下一步生成阶段会转成 (1, seq_len) 的张量
input_token_ids = tokenizer.encode(prompt)
text = tokenizer.decode(input_token_ids)
text

5. Generate text

In [ ]:
# ============================================================
# 第五部分：贪心解码（greedy decoding）生成文本，逐 token 以生成器(yield)方式流式输出
# 注意：这是「无 KV 缓存」的最简实现——每一步都把当前完整的 token_ids 重新喂给模型做一次
# 完整前向传播，不会缓存之前算过的 K/V，效率较低，仅用于演示，不适合真实高吞吐推理场景。
# ============================================================
def generate_text_basic_stream(model, token_ids, max_new_tokens, eos_token_id=None):

    # 切换到 eval 模式（关闭 dropout 等训练专用行为；本实现中的 attention_dropout 实际也未被使用）
    model.eval()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            # token_ids: (batch, cur_len) -> 前向输出 logits: (batch, cur_len, vocab_size)，
            # 只取最后一个位置的 logits (batch, vocab_size) 用于预测下一个 token
            out = model(token_ids)[:, -1]
            # 贪心解码：直接取概率最大的 token（不做采样），next_token 形状 (batch, 1)
            next_token = torch.argmax(out, dim=-1, keepdim=True)

            # 如果 batch 中所有样本都生成了结束符，就提前停止生成
            if (eos_token_id is not None
                   and torch.all(next_token == eos_token_id)):
               break

            yield next_token

            # 把新生成的 token 拼接到序列末尾，下一轮会把完整序列重新喂给模型（无 KV 缓存的代价）
            token_ids = torch.cat([token_ids, next_token], dim=1)

# 把编码好的一维 token id 列表转成 (1, seq_len) 的张量，batch_size=1，并放到目标设备上
input_token_ids_tensor = torch.tensor(input_token_ids, device=device).unsqueeze(0)


# 若使用 CUDA，重置显存峰值统计，便于之后测量本次生成实际占用的最大显存
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()


# 开始流式生成，每次循环拿到一个新 token 就立刻解码并打印，形成打字机效果
for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=500,
    eos_token_id=tokenizer.eos_token_id
):
    # token: (1, 1) -> squeeze(0) 变成 (1,) -> tolist() 转成 Python list，交给分词器 decode
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )


# 生成结束后打印本次推理占用的 GPU 显存峰值，便于评估不同模型规模/上下文长度的显存需求
if torch.cuda.is_available():
    def calc_gpu_gb(x):
        return f"{x / 1024 / 1024 / 1024:.2f} GB"

    print(f"\n\nGPU memory used: {calc_gpu_gb(torch.cuda.max_memory_allocated())}")